In [ ]:
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import get_prop_improve, get_param_vals_by_analysis, get_prop_improve
from emu_renewal.plotting import MOB_SOURCE_MAP, MOB_COLOURS, ANALYSIS_TYPES, ANALYSIS_NAMES, plot_world_country_outline, plot_prop_improve

plt.style.use("default")
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "45878514"
countries = ls(job_path)
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in countries}

world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)

# Best analysis
Best selected approach to analysis as determined
by the lowest average dispersion value needed in model fitting.
Google, green; Facebook tiles visited, blue; Facebook single tile, red.

In [ ]:
best_mob = {c: disp_posts[c].mean().idxmin() for c in countries}
world["best_mob"] = world["ISO_A3"].map(best_mob)
world["best_mob_colour"] = world["best_mob"].map(MOB_COLOURS | {"no_mob": "0.45"})
world.loc[world["best_mob_colour"].isna(), "best_mob_colour"] = "0.975"

fig, ax = plot_world_country_outline(world)
ax.set_title("Best mobility type")
world.plot(ax=ax, color=world["best_mob_colour"]);

# Improvement over no mobility analysis for each mobility type
The following plots show the proportion of analyses for which the
dispersion value was lower than a randomly selected equivalent run
from the analysis with no mobility data implemented.
Red colours indicate implementation of mobility reduced the 
dispersion parameter required for fitting.
Intermediate and blue colours indicate this did not occur.

In [ ]:
analysis_type = "g_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve, analysis_type, world)

In [ ]:
analysis_type = "fb_visited_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve, analysis_type, world)

In [ ]:
analysis_type = "fb_singletile_mob"
display(Markdown(f"## {ANALYSIS_NAMES[analysis_type]}"))
prop_improve = get_prop_improve(disp_posts, analysis_type)
plot_prop_improve(prop_improve, analysis_type, world)